# Cặp ảnh tối thật (LOL) cho p47_restore

Notebook này **không train**. Nó gom các cặp ảnh **chụp thật** "tối → đủ sáng" của LOL (Wei và cs., BMVC 2018) và, nếu gắn,
LOL-v2 Real_captured (Yang và cs., TIP 2021) về đúng bố cục mà repo đọc (`qjepa/data/scene_pairs.py`):

```
<out>/index.csv     file, split (train / val), scene, scene_split, flare (= nguồn: lol_v1 / lol_v2_real), kind (= lowlight)
<out>/gt/<file>     ảnh đủ sáng (đích)
<out>/input/<file>  cùng cảnh, chụp tối (đầu vào)
```

Cột `kind = lowlight` để Cell 4 của `qwt-jaco-jepa-ijepa.ipynb` phân biệt bộ này với bộ VisDrone cùng bố cục.

**Nguồn**, theo thứ tự:
1. LOL-v1 (`our485/{low,high}` → train, `eval15/{low,high}` → val) hoặc LOL-v2 Real (`Real_captured/Train/{Low,Normal}` → train,
   `Test/{Low,Normal}` → val) đã gắn làm Input, ở độ sâu bất kỳ;
2. không có gì thì **tự tải LOL-v1** (485 + 15 cặp, ~347 MB) từ bản sao công khai trên Hugging Face
   `geekyrakshit/LoL-Dataset` — cần bật **Internet** (Settings → Internet on).

LOL-v2 Real chứa lại một phần ảnh của LOL-v1: cặp trùng (so bằng ảnh xám 16 × 16) chỉ giữ một bản, và ảnh nằm trong val của bộ
nào thì ở val, không lọt vào train qua bộ kia.

**Sau khi chạy xong:** Save Version (Save & Run All) → tab Output → **New Dataset**, đặt tên ví dụ `lowlight-pairs`
(thư mục `lowlight-pairs/` trong Output). Gắn dataset đó vào notebook train; Cell 4 tự tìm.

## Cell 1 — Lựa chọn

In [ ]:
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/lowlight-pairs')         # thu muc se thanh dataset
DOWNLOAD_DIR = Path('/tmp/lol_download')             # ngoai /kaggle/working: khong lot vao Output
DOWNLOAD_IF_MISSING = True                           # khong thay LOL trong Input thi tai LOL-v1 tu Hugging Face
LOL_V1_URL = 'https://huggingface.co/datasets/geekyrakshit/LoL-Dataset/resolve/main/lol_dataset.zip'
PREVIEW = 8                                          # so cap ve o Cell 4

## Cell 2 — Hàm dùng chung (chỉ định nghĩa, chưa chạy gì)

In [ ]:
import csv
import hashlib
import shutil
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
from PIL import Image

IMAGE_SUFFIXES = ('.png', '.jpg', '.jpeg', '.bmp')


def _images(folder):
    return {p.name: p for p in sorted(folder.iterdir()) if p.suffix.lower() in IMAGE_SUFFIXES} if folder.is_dir() else {}


def _child(folder, *names):
    """Thu muc con dau tien trong ``names`` (khong phan biet hoa thuong), None neu khong co."""
    if not folder.is_dir():
        return None
    lower = {p.name.lower(): p for p in folder.iterdir() if p.is_dir()}
    return next((lower[n.lower()] for n in names if n.lower() in lower), None)


def find_sources(base, depth=7):
    """Moi bo LOL trong ``base``: [(ten nguon, split, thu muc toi, thu muc sang, cach ghep ten)].

    LOL-v1: our485/{low,high} (train), eval15/{low,high} (val), cung ten file.
    LOL-v2 Real_captured: Train/{Low,Normal}, Test/{Low,Normal}; lowXXXXX.png <-> normalXXXXX.png."""
    base, found = Path(base), []
    seen = set()
    for level in range(depth):
        for folder in (base.glob('/'.join(['*'] * level)) if level else [base]):
            folder = folder.resolve()
            if not folder.is_dir() or folder in seen:
                continue
            seen.add(folder)
            for name, split in (('our485', 'train'), ('eval15', 'val')):
                part = _child(folder, name)
                low, high = (_child(part, 'low'), _child(part, 'high')) if part else (None, None)
                if low and high:
                    found.append(('lol_v1', split, low, high, 'same'))
            real = _child(folder, 'Real_captured')
            for name, split in (('Train', 'train'), ('Test', 'val')):
                part = _child(real, name) if real else None
                low, high = (_child(part, 'Low'), _child(part, 'Normal')) if part else (None, None)
                if low and high:
                    found.append(('lol_v2_real', split, low, high, 'low_normal'))
    return found


def download_lol_v1(target):
    """LOL-v1 (485 + 15 cap) tu ban sao cong khai tren Hugging Face (geekyrakshit/LoL-Dataset, ~347 MB).
    Notebook phai bat Internet (Settings -> Internet on)."""
    target = Path(target)
    archive = target / 'lol_dataset.zip'
    if not (target / 'lol_dataset').is_dir():
        target.mkdir(parents=True, exist_ok=True)
        if not archive.is_file():
            urllib.request.urlretrieve(LOL_V1_URL, archive)
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(target)
        archive.unlink()
    return target


def _partner(name, rule):
    return name if rule == 'same' else 'normal' + name[3:] if name.lower().startswith('low') else None


def _fingerprint(path):
    """16 x 16 xam lam tron: cung mot anh du luu lai o dinh dang/do nen khac (LOL-v2 Real chua lai anh cua LOL-v1)."""
    with Image.open(path) as image:
        small = np.asarray(image.convert('L').resize((16, 16), Image.Resampling.BOX), dtype=np.float32)
    return hashlib.sha1(np.round(small / 8).astype(np.uint8).tobytes()).hexdigest()


def collect_pairs(sources):
    """[(nguon, split, anh toi, anh sang)], bo cap lech co va cap trung; anh nao o val cua bat ky bo nao thi o val."""
    pairs, problems = [], []
    for source, split, low_dir, high_dir, rule in sources:
        highs = _images(high_dir)
        for name, low in _images(low_dir).items():
            high = highs.get(_partner(name, rule) or '')
            if high is None:
                problems.append(f'{source}/{split}: {name} khong co anh sang tuong ung')
                continue
            with Image.open(low) as a, Image.open(high) as b:
                if a.size != b.size:
                    problems.append(f'{source}/{split}: {name} co khac nhau {a.size} vs {b.size}')
                    continue
            pairs.append((source, split, low, high))
    keyed = {}
    for source, split, low, high in pairs:
        key = _fingerprint(high)
        keep = keyed.get(key)
        # Trung: giu mot ban; val thang train (khong de anh val lot vao train qua bo kia).
        if keep is None or (split == 'val' and keep[1] == 'train'):
            keyed[key] = (source, split, low, high)
    unique = sorted(keyed.values(), key=lambda row: (row[0], row[1], row[2].name))
    return unique, problems, len(pairs) - len(unique)


def write_dataset(pairs, out):
    """out/gt/<ten>.png (sang), out/input/<ten>.png (toi), out/index.csv (file, split, scene, scene_split, flare, kind)."""
    out = Path(out)
    shutil.rmtree(out, ignore_errors=True)
    (out / 'gt').mkdir(parents=True)
    (out / 'input').mkdir()
    rows = []
    for source, split, low, high in pairs:
        name = f'{source}_{split}_{Path(low.name).stem}.png'
        for folder, path in (('input', low), ('gt', high)):
            with Image.open(path) as image:
                image.convert('RGB').save(out / folder / name)
        rows.append({'file': name, 'split': split, 'scene': f'{source}/{split}/{low.name}', 'scene_split': split,
                     'flare': source, 'kind': 'lowlight'})
    with (out / 'index.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, ['file', 'split', 'scene', 'scene_split', 'flare', 'kind'])
        writer.writeheader()
        writer.writerows(rows)
    return rows

## Cell 3 — Tìm nguồn, ghép cặp, ghi dataset

In [ ]:
import time

started = time.perf_counter()
sources = find_sources(INPUT) if INPUT.is_dir() else []
if not sources and DOWNLOAD_IF_MISSING:
    print('Không thấy LOL trong Input -> tải LOL-v1 từ Hugging Face (~347 MB)...', flush=True)
    sources = find_sources(download_lol_v1(DOWNLOAD_DIR))
assert sources, ('Không có cặp LOL nào: gắn LOL-v1 / LOL-v2 làm Input, hoặc bật Internet và DOWNLOAD_IF_MISSING = True.')
for source, split, low, high, _ in sources:
    print(f'{source:12s} {split:5s} {low}  <->  {high}')
pairs, problems, duplicates = collect_pairs(sources)
for line in problems[:20]:
    print('  bỏ:', line)
rows = write_dataset(pairs, OUT)
counts = {}
for row in rows:
    counts.setdefault((row['flare'], row['split']), 0)
    counts[(row['flare'], row['split'])] += 1
print(f'\n{len(rows)} cặp -> {OUT} | {duplicates} cặp trùng đã gộp | {len(problems)} file bỏ')
for (source, split), count in sorted(counts.items()):
    print(f'  {source:12s} {split:5s} {count}')
assert any(row['split'] == 'val' for row in rows) and any(row['split'] == 'train' for row in rows)
print('Cell 3 giây:', round(time.perf_counter() - started, 1))

## Cell 4 — Xem vài cặp và độ sáng

Mỗi cặp: **vào (tối)** | **đích (đủ sáng)**. Bảng in độ sáng trung bình (luma sRGB) của hai bên: đích LOL thường quanh 0,4–0,5,
ảnh tối quanh 0,05. Repo làm sáng thêm đích nào dưới 0,30 (`phase2.target_brighten`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def luma(path):
    with Image.open(path) as image:
        rgb = np.asarray(image.convert('RGB'), dtype=np.float32) / 255
    return float((rgb @ np.array([0.299, 0.587, 0.114], dtype=np.float32)).mean())


for split in ('train', 'val'):
    chosen = [row for row in rows if row['split'] == split]
    dark = [luma(OUT / 'input' / row['file']) for row in chosen[:200]]
    bright = [luma(OUT / 'gt' / row['file']) for row in chosen[:200]]
    print(f'{split:5s} n={len(chosen):4d} | độ sáng vào {np.mean(dark):.3f} | đích {np.mean(bright):.3f}')

shown = rows[:: max(1, len(rows) // PREVIEW)][:PREVIEW]
figure, axes = plt.subplots(len(shown), 2, figsize=(8, 2.8 * len(shown)), squeeze=False)
for (left, right), row in zip(axes, shown):
    for axis, folder in ((left, 'input'), (right, 'gt')):
        axis.imshow(Image.open(OUT / folder / row['file']).convert('RGB'))
        axis.set_title(f"{row['split']} {row['file']} ({'vào' if folder == 'input' else 'đích'})", fontsize=8)
        axis.axis('off')
figure.tight_layout()
plt.show()